<a href="https://colab.research.google.com/github/tiran543/Statistical-Learning-e23094/blob/main/Copy_of_data_wrangling_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson’s $r$.
    * **Categorical-Categorical**: Cramér’s $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

In [ ]:
!pip install "git+https://github.com/e23250-max/Statistical-Learning-e23250.git#subdirectory=data-analysis-tool"

  Cloning https://github.com/e23250-max/Statistical-Learning-e23250.git to /tmp/pip-req-build-43t_vy_p
  Running command git clone --filter=blob:none --quiet https://github.com/e23250-max/Statistical-Learning-e23250.git /tmp/pip-req-build-43t_vy_p
  Resolved https://github.com/e23250-max/Statistical-Learning-e23250.git to commit 9ce54f6f13188c580ec2c99b4497a8aeaae10b8b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for data-analysis-tool: filename=data_analysis_tool-0.1.0-py3-none-any.whl size=4477 sha256=2795211524062e1aed791aea1222d5e2c813a5cbf020343b2609ec1d78233ea8
  Stored in directory: /tmp/pip-ephem-wheel-cache-50fm168x/wheels/30/e5/09/d557777c16714339e937beaf4afdcd0b110d9cafdc872cc330
Successfully built data-analysis-tool


In [ ]:
import pandas as pd
from data_analysis import DataInspector

print("--- 1. INITIALIZING & LOADING ---")
inspector = DataInspector()
# Using the Titanic dataset via URL so you don't have to manually upload a file
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
inspector.df = pd.read_csv(url)
inspector._initial_cleanup()

print("\n--- 2. STRUCTURAL SUMMARY ---")
inspector.get_summary()

print("\n--- 3. CLEANING ---")
inspector.remove_duplicates()
inspector.handle_missing_values(strategy='median')
inspector.handle_outliers(columns=['Fare'], delete=False)

print("\n--- 4. FEATURE ENGINEERING ---")
final_ml_df = inspector.create_normalized_data_df()
display(final_ml_df.head())

print("\n--- 5. VISUALIZATIONS ---")
inspector.plot_numerical(['Age', 'Fare'])
inspector.plot_relationship('Survived', 'Age')
inspector.plot_all_associations_heatmap()

--- 1. INITIALIZING & LOADING ---

--- 2. STRUCTURAL SUMMARY ---
Dataset Shape: 891 Rows, 12 Columns
Numerical Columns (7): ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical Columns (5): ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']

First 20 Rows Preview:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C



--- 3. CLEANING ---
Removed 0 duplicate rows.
Missing values handled using 'median' strategy.
Identified 116 outliers in Fare (Bounds: -26.724 to 65.6344).

--- 4. FEATURE ENGINEERING ---


/usr/local/lib/python3.12/dist-packages/data_analysis/core.py:106: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df[col].fillna(self.df[col].median(), inplace=True)


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,"Name_Abbott, Mr. Rossmore Edward","Name_Abbott, Mrs. Stanton (Rosa Hunt)","Name_Abelson, Mr. Samuel",...,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_Q,Embarked_S
0,-1.729137,-0.788829,0.826913,-0.565419,0.432550,-0.473408,-0.502163,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1,-1.725251,1.266279,-1.565228,0.663488,0.432550,-0.473408,0.786404,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,-1.721365,1.266279,0.826913,-0.258192,-0.474279,-0.473408,-0.488580,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,-1.717480,1.266279,-1.565228,0.433068,0.432550,-0.473408,0.420494,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,-1.713594,-0.788829,0.826913,0.433068,-0.474279,-0.473408,-0.486064,False,False,False,...,False,False,False,False,False,False,False,False,False,True



--- 5. VISUALIZATIONS ---
